In [2]:
import warnings

import pandas as pd

warnings.filterwarnings("ignore")

from features import build_features

print("Loading data...")

df = pd.read_parquet("datasets/train_clean.parquet")
df["event_date"] = pd.to_datetime(df["event_date"])

print(f"  Shape: {df.shape}, users: {df['user_id'].nunique()}")

SPLITS = {
    "split1": {
        "feat_cutoff": pd.Timestamp("2025-11-13"),
        "tgt_start": pd.Timestamp("2025-11-14"),
        "tgt_end": pd.Timestamp("2025-12-13"),
    },
    "split2": {
        "feat_cutoff": pd.Timestamp("2025-12-13"),
        "tgt_start": pd.Timestamp("2025-12-14"),
        "tgt_end": pd.Timestamp("2026-01-13"),
    },
    "split3": {
        "feat_cutoff": pd.Timestamp("2026-01-13"),
        "tgt_start": pd.Timestamp("2026-01-14"),
        "tgt_end": pd.Timestamp("2026-02-13"),
    },
}

FEATURES_GROUP = ["all_history", "rolling_windows", "velocity", "trend", "gmv", "conversion_rates", "ewm"]


def build_target(tgt_start: pd.Timestamp, tgt_end: pd.Timestamp) -> pd.DataFrame:
    mask = (df["event_date"] >= tgt_start) & (df["event_date"] <= tgt_end)
    return (
        df[mask]
        .groupby("user_id")
        .agg(target=("gmv", "sum"))
        .reset_index()
    )


def attach_target(feats: pd.DataFrame, target_df: pd.DataFrame) -> pd.DataFrame:
    all_users = pd.DataFrame({"user_id": df["user_id"].unique()})
    merged = all_users.merge(feats, on="user_id", how="left").fillna(0)
    merged = merged.merge(target_df, on="user_id", how="left")
    merged["target"] = merged["target"].fillna(0)
    return merged


def save(frame: pd.DataFrame, split_name: str, feature) -> None:
    path = f"datasets/{split_name}_{feature}.parquet"
    frame.to_parquet(path, index=False)
    print(f"  Saved {path}  ({frame.shape[0]} rows, {frame.shape[1]} cols)")

  Shape: (30631006, 18), users: 250000


In [5]:
from sklearn.model_selection import train_test_split

cfg = SPLITS["split3"]
target_df = build_target(cfg["tgt_start"], cfg["tgt_end"])

for f in FEATURES_GROUP:
    feature_flags = {name: False for name in FEATURES_GROUP}
    feature_flags[f] = True

    feats = build_features(df, cfg["feat_cutoff"], **feature_flags)
    frame = attach_target(feats, target_df)
    train, test = train_test_split(frame, test_size=0.2, random_state=42)
    save(train, "train", f)
    save(test, "test", f)

  Saved datasets/train_ewm.parquet  (200000 rows, 25 cols)
  Saved datasets/test_ewm.parquet  (50000 rows, 25 cols)


In [10]:
from sklearn.model_selection import train_test_split

cfg = SPLITS["split3"]
target_df = build_target(cfg["tgt_start"], cfg["tgt_end"])

feature_flags = {name: False for name in FEATURES_GROUP}

feats = build_features(df, cfg["feat_cutoff"], all_history=True, rolling_windows=True, velocity=True, trend=True, gmv=True)
frame = attach_target(feats, target_df)
train, test = train_test_split(frame, test_size=0.2, random_state=42)
save(train, "train", "all_history_rolling_windows_velocity_trend_gmv")
save(test, "test", "all_history_rolling_windows_velocity_trend_gmv")

  Saved datasets/test_all_history_rolling_windows_velocity_trend_gmv.parquet  (50000 rows, 104 cols)


In [1]:
for split_name, cfg in SPLITS.items():
    for f in FEATURES_GROUP:
        feature_flags = {name: False for name in FEATURES_GROUP}
        feature_flags[f] = True

        feats = build_features(df, cfg["feat_cutoff"], **feature_flags)
        target_df = build_target(cfg["tgt_start"], cfg["tgt_end"])
        full = attach_target(feats, target_df)

        save(full, f"{split_name}", f)

NameError: name 'SPLITS' is not defined

In [3]:
for split_name, cfg in SPLITS.items():
    feats = build_features(df, cfg["feat_cutoff"], all_history=True, rolling_windows=True, velocity=True, trend=True, gmv=True)
    target_df = build_target(cfg["tgt_start"], cfg["tgt_end"])
    full = attach_target(feats, target_df)

    save(full, f"{split_name}", "all_history_rolling_windows_velocity_trend_gmv")

  Saved datasets/split3_all_history_rolling_windows_velocity_trend_gmv.parquet  (250000 rows, 104 cols)


In [5]:
for split_name, cfg in SPLITS.items():
    feats = build_features(df, cfg["feat_cutoff"])
    target_df = build_target(cfg["tgt_start"], cfg["tgt_end"])
    full = attach_target(feats, target_df)

    save(full, f"{split_name}", "base")

  Saved datasets/split3_base.parquet  (250000 rows, 22 cols)


In [7]:
for split_name, cfg in SPLITS.items():
    feature_flags = {name: True for name in FEATURES_GROUP}

    feats = build_features(df, cfg["feat_cutoff"], **feature_flags)
    target_df = build_target(cfg["tgt_start"], cfg["tgt_end"])
    full = attach_target(feats, target_df)

    save(full, f"{split_name}", "all_features")

print("\nDone.")

  Saved datasets/split3_all_features.parquet  (250000 rows, 110 cols)

Done.


In [11]:
sub_cutoff = df["event_date"].max()

sub = build_features(df, sub_cutoff, all_history=True, rolling_windows=True, velocity=True, trend=True, gmv=True)

save(sub, "submission", "all_history_rolling_windows_velocity_trend_gmv")

  Saved datasets/submission_all_history_rolling_windows_velocity_trend_gmv.parquet  (250000 rows, 103 cols)
